# V7.1 — YOLO26m fine-tune trên curated + 171126 (RTX 4060)

Fine-tune từ checkpoint seed85 curated. Dataset local giữ nguyên val/test, chỉ bổ sung dữ liệu 171126 vào train. Checkpoint được lưu local mỗi epoch và đồng bộ Camber mỗi 5 epoch.

In [ ]:
# Cấu hình chính
from pathlib import Path
import os, shutil, subprocess, sys, json, time

RUN_NAME = 'yolo26m_finetune'
EXPERIMENT = 'finding_v7p1_171126'
REMOTE_ROOT = f'stash://bunpmc/projects/{EXPERIMENT}'
REMOTE_RUN = f'{REMOTE_ROOT}/{RUN_NAME}'
SOURCE_WEIGHTS = 'stash://bunpmc/projects/finding_seed/seed85/curated_yolo26m/weights/best.pt'

EPOCHS = 30          # 30 epoch mới, tính từ seed85 best
IMGSZ = 640
BATCH = 4           # an toàn cho đa số RTX 4060 8 GB; tăng 8 nếu VRAM chịu được
SEED = 85
SYNC_EVERY = 5

def find_repo():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir():
            return p
    raise RuntimeError('Hãy mở notebook từ repo C:/Users/intern/Documents/SEP490/train')

ROOT = find_repo()
DATA_YAML = ROOT / 'benchmakring_transmission_line_defect_model/data/finding_nemo_human_reviewed_6cls_v2_171126/data.yaml'
PROJECT = ROOT / 'runs' / EXPERIMENT
RUN_DIR = PROJECT / RUN_NAME
INIT_WEIGHT = ROOT / 'weights' / 'v7p1_seed85_curated_yolo26m_best.pt'
PROJECT.mkdir(parents=True, exist_ok=True)
INIT_WEIGHT.parent.mkdir(parents=True, exist_ok=True)
assert DATA_YAML.is_file(), DATA_YAML
print('ROOT       :', ROOT)
print('DATA       :', DATA_YAML)
print('LOCAL RUN  :', RUN_DIR)
print('CAMBER RUN :', REMOTE_RUN)

## 1. Môi trường
Nếu import Ultralytics lỗi, bỏ comment dòng `%pip` rồi chạy cell và restart kernel. Giữ phiên bản cố định để lần resume không đổi môi trường.

In [ ]:
# %pip install -U "ultralytics==8.4.154" python-dotenv
import torch, ultralytics
from ultralytics import YOLO
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass
assert torch.cuda.is_available(), 'Không thấy CUDA. Kiểm tra PyTorch CUDA/driver NVIDIA.'
print('Ultralytics:', ultralytics.__version__)
print('PyTorch    :', torch.__version__)
print('GPU        :', torch.cuda.get_device_name(0))
print('VRAM       :', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

## 2. Kết nối Camber và tải checkpoint
Cần `CAMBER_API_KEY` trong `.env` hoặc Camber CLI đã đăng nhập. Cell không in API key.

In [ ]:
def find_camber():
    candidates = [
        os.environ.get('CAMBER_BIN'), shutil.which('camber'),
        str(Path.home() / '.camber/bin/camber.exe'),
        str(Path.home() / '.camber/bin/camber'),
    ]
    for item in candidates:
        if item and Path(item).is_file():
            return str(Path(item))
    raise RuntimeError('Không tìm thấy Camber CLI. Cài CLI hoặc đặt CAMBER_BIN trong .env')

CAMBER = find_camber()
CAMBER_ENV = os.environ.copy()

def camber(*parts, check=True):
    p = subprocess.run([CAMBER, 'stash', *map(str, parts)], env=CAMBER_ENV, capture_output=True, text=True)
    if check and p.returncode:
        raise RuntimeError((p.stderr or p.stdout)[-2000:])
    return p

auth = subprocess.run([CAMBER, 'me'], env=CAMBER_ENV, capture_output=True, text=True)
assert auth.returncode == 0, (auth.stderr or auth.stdout)[-1200:]
print('Camber authentication OK:', CAMBER)

def ensure_remote_dir(uri):
    # mkdir có thể báo đã tồn tại; xác nhận bằng ls.
    if camber('ls', uri, check=False).returncode != 0:
        made = camber('mkdir', uri, check=False)
        if made.returncode != 0 and camber('ls', uri, check=False).returncode != 0:
            raise RuntimeError((made.stderr or made.stdout)[-1200:])

for uri in ('stash://bunpmc/projects', REMOTE_ROOT, REMOTE_RUN, REMOTE_RUN + '/weights'):
    ensure_remote_dir(uri)

if not INIT_WEIGHT.is_file():
    print('Downloading initial seed85 best.pt ...')
    camber('cp', SOURCE_WEIGHTS, INIT_WEIGHT)
print('Initial checkpoint:', INIT_WEIGHT, round(INIT_WEIGHT.stat().st_size / 2**20, 1), 'MiB')

## 3. Restore tiến trình nếu máy từng bị ngắt
Nếu local đã có `last.pt`, notebook ưu tiên local. Nếu chưa có, nó thử tải checkpoint gần nhất từ Camber.

In [ ]:
def try_restore(remote, local):
    if local.exists():
        return True
    local.parent.mkdir(parents=True, exist_ok=True)
    p = camber('cp', remote, local, check=False)
    if p.returncode == 0 and local.exists():
        print('[RESTORED]', local)
        return True
    return False

last_pt = RUN_DIR / 'weights/last.pt'
if not last_pt.exists():
    try_restore(REMOTE_RUN + '/weights/last.pt', last_pt)
    try_restore(REMOTE_RUN + '/results.csv', RUN_DIR / 'results.csv')
    try_restore(REMOTE_RUN + '/args.yaml', RUN_DIR / 'args.yaml')
print('Resume checkpoint:', last_pt if last_pt.exists() else 'không có — bắt đầu fine-tune mới')

## 4. Train / resume
`last.pt` được ghi mỗi epoch. Cứ 5 epoch sẽ upload `last.pt`, `best.pt`, `results.csv`, `args.yaml` và checkpoint `epochN.pt` lên Camber.

In [ ]:
def sync_file(local: Path, remote: str):
    if local.is_file():
        camber('cp', local, remote)

def sync_progress(epoch=None, final=False):
    ensure_remote_dir(REMOTE_RUN); ensure_remote_dir(REMOTE_RUN + '/weights')
    for rel in ('results.csv', 'args.yaml'):
        sync_file(RUN_DIR / rel, REMOTE_RUN + '/' + rel)
    for rel in ('weights/last.pt', 'weights/best.pt'):
        sync_file(RUN_DIR / rel, REMOTE_RUN + '/' + rel)
    if epoch:
        rel = f'weights/epoch{epoch}.pt'
        sync_file(RUN_DIR / rel, REMOTE_RUN + '/' + rel)
    status = {
        'experiment': EXPERIMENT, 'run': RUN_NAME, 'epoch': epoch, 'target_epochs': EPOCHS,
        'final_sync': final, 'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'dataset': str(DATA_YAML), 'source_weights': SOURCE_WEIGHTS,
    }
    status_path = RUN_DIR / 'camber_status.json'
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
    sync_file(status_path, REMOTE_RUN + '/camber_status.json')
    print(f'[CAMBER SYNC] epoch={epoch} -> {REMOTE_RUN}')

resume = last_pt.is_file()
model = YOLO(str(last_pt if resume else INIT_WEIGHT))

def on_fit_epoch_end(trainer):
    # Ultralytics đã chạy validation ở cuối mỗi epoch; in một dòng gọn để dễ theo dõi.
    m = trainer.metrics or {}
    def value(key):
        v = m.get(key, float('nan'))
        try: return f'{float(v):.4f}'
        except Exception: return str(v)
    elapsed = (time.time() - train_clock['start']) / 60
    print(f'\n[V7.1 PROGRESS] epoch {trainer.epoch + 1}/{trainer.epochs} | '
          f'val mAP50={value("metrics/mAP50(B)")} | '
          f'val mAP50-95={value("metrics/mAP50-95(B)")} | '
          f'P={value("metrics/precision(B)")} R={value("metrics/recall(B)")} | '
          f'{elapsed:.1f} min elapsed', flush=True)

def on_model_save(trainer):
    epoch = trainer.epoch + 1
    if epoch % SYNC_EVERY == 0 or epoch == trainer.epochs:
        sync_progress(epoch=epoch, final=(epoch == trainer.epochs))

train_clock = {'start': time.time()}
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
model.add_callback('on_model_save', on_model_save)

if resume:
    print('RESUME:', last_pt)
    results = model.train(resume=True)
else:
    print('START fine-tune từ:', INIT_WEIGHT)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
        optimizer='AdamW', lr0=1e-4, lrf=0.01, cos_lr=True, warmup_epochs=1.0,
        momentum=0.937, weight_decay=5e-4,
        mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0,
        translate=0.1, scale=0.5, degrees=0.0, shear=0.0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.4,
        amp=True, seed=SEED, deterministic=True, workers=4, cache='disk',
        project=str(PROJECT), name=RUN_NAME, exist_ok=True,
        save=True, save_period=SYNC_EVERY, plots=True, patience=12, val=True, verbose=True,
    )
sync_progress(epoch=None, final=True)
print('Training finished:', RUN_DIR)

## 5. Xem best epoch và test cố định

In [ ]:
import pandas as pd
csv_path = RUN_DIR / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
metric = 'metrics/mAP50-95(B)'
best_i = df[metric].idxmax()
display(df.loc[[best_i], ['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', metric]])

best = RUN_DIR / 'weights/best.pt'
assert best.is_file(), best
test_result = YOLO(str(best)).val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=0, plots=True)
print('Test mAP50   :', test_result.box.map50)
print('Test mAP50-95:', test_result.box.map)
sync_progress(epoch=None, final=True)

### Lưu ý
- Không xóa thư mục `runs/finding_v7p1_171126/yolo26m_finetune` khi muốn resume.
- Nếu OOM với batch 4, xóa run chưa hoàn chỉnh, đặt `BATCH=2` rồi train lại từ đầu. Không đổi batch giữa một lần resume.
- Kết quả test chỉ chạy sau khi đã chọn `best.pt` bằng val; không dùng test để chọn epoch.